In [1]:
#!pip install sympy

In [2]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from Sympy_OPF_LALM_class import extract_qhd_solution_vector
from Sympy_OPF_LALM_class import initialize_qhd_acopf_log
from Sympy_OPF_LALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json

In [3]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # Convert "k1", "k2", ... into integer keys 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens


In [4]:
# Initialize the model.
# model = SympyACOPFModel()   # Enable the proximal option if needed later.
n = 3 # Select the test system size: 2, 3, or larger.

if n == 2:
    # 2-bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3-bus model
    model = SympyACOPFModel()

else:
    # n-bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 3 buses 3 lines and 2 generators.


In [5]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 50
alpha = 0.3   # 对偶步长尽量小一点 simi 0.5-1.0 (2 bus)
max_outer = 600
tol = 1e-4
option = 1 # 1: QHD, 2: Gurobi
qhd_solver = "simbi"  # simbi / openjij / gurobi

# ========= 新增：初始化日志文件 =========
# alpha 调度选项
# alpha_option: "fixed" / "decay"
alpha_option = "decay"
fixed_alpha = alpha

# 若 alpha_option == "decay"，则有两种输入方式：
# 1. alpha0 + alpha_decay_factor -> alpha_(k+1) = alpha_k * alpha_decay_factor
# 2. alpha0 + alphaN -> alpha_decay_factor = (alphaN / alpha0)**(1 / max_outer)
# 若用户什么都没输入，则默认采用方案 2：alpha0=1, alphaN=1e-4
alpha0 = 10#None
alpha_decay_factor = None
alphaN = 1e-7 #None

if alpha_option == "fixed":
    if fixed_alpha <= 0:
        raise ValueError("fixed_alpha must be positive.")
    alpha = fixed_alpha
elif alpha_option == "decay":
    alpha0_use = 1.0 if alpha0 is None else alpha0
    if alpha0_use <= 0:
        raise ValueError("alpha0 must be positive.")
    if alpha_decay_factor is not None:
        if alpha_decay_factor <= 0:
            raise ValueError("alpha_decay_factor must be positive.")
        decay_factor = alpha_decay_factor
    else:
        alphaN_use = 1e-4 if alphaN is None else alphaN
        if alphaN_use <= 0:
            raise ValueError("alphaN must be positive.")
        decay_factor = (alphaN_use / alpha0_use) ** (1.0 / max_outer)
    alpha = alpha0_use
else:
    raise ValueError("alpha_option must be 'fixed' or 'decay'.")

log_file = initialize_qhd_acopf_log(model, folder="logs", option=option, qhd_solver=qhd_solver)
print("Log file:", log_file)
print(f"Alpha option: {alpha_option}")
if alpha_option == "fixed":
    print(f"Fixed alpha: {fixed_alpha}")
else:
    print(f"Decay alpha: alpha0={alpha0_use}, decay_factor={decay_factor}")

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    if alpha_option == "fixed":
        alpha = fixed_alpha
    else:
        alpha = alpha0_use * (decay_factor ** k)

    print(f"\n--- Outer Iteration {k} ---")
    print(f"alpha = {alpha:.6e}")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=8,
                agents=1024,
                max_steps=5000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                #ballistic=True,
                #heated=True,
                best_only=False,
                verbose=True
            )
        elif qhd_solver == "openjij":
            qhd_model.openjij_setup(
                resolution=6,
                shots=2048,
                sampler_name="SQASampler",
                post_processing_method='TNC',
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 10000,
                    "reinitialize_state": True,
                },
            )
        elif qhd_solver == "gurobi":
            qhd_model.gurobi_setup(
                resolution=4,
                shots=20,
                embedding_scheme="unary",
                solver_mode="ising",
                time_limit=30,
                threads=0,
                log_to_console=False,
                post_processing_method='TNC',
            )
        else:
            raise ValueError(f"Unsupported qhd_solver={qhd_solver!r}. Use 'simbi', 'openjij', or 'gurobi'.")
        response = qhd_model.optimize(refine=True, verbose=0)
        #response = qhd_model.optimize(refine=False, verbose=0)

        x_new = extract_qhd_solution_vector(response, expected_len=len(variable_list))

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=alpha,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    # Keep rho fixed for stability diagnostics.
    print("Keeping rho fixed at", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(x_new)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    log_file = PrintQHDACOPFResults(
        model,
        x_new,
        log_file=log_file,
        iteration=k,
        folder="logs",
        print_to_console=True,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-3_04-20-2026_12-11-43.txt
Alpha option: decay
Decay alpha: alpha0=10, decay_factor=0.9697653591082493

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---
alpha = 1.000000e+01


Bifurcated agents:   0%|          | 0/1024 [00:02<?, ?it/s]


Minimizer: [0.    0.    0.125 0.125 0.875 0.875 0.875 0.5   0.5   0.5   0.125 0.25
 0.25  0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
 0.5   0.    0.    0.    0.    0.    0.   ]
||h(x)|| = 3.5413675999642606
[rho-check] ||h_old||=4.622e-01, ||h_new||=3.541e+00, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-20 12:12:20
Iteration: 0
rho: 50
alpha: 10
objective_value: 0
feasible: False
max_abs_h: 2.617443136269e+00
l2_norm_h: 3.541367599964e+00
lambda_inf_norm: 2.617443136269e+01
lambda_l2_norm: 3.541367599964e+01
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.999746	-0.000096	0.998709	0.000000	1.805442	0.000000	0.000000
2	0.909605	0.029761	0.903918	0.000762	-1.862793	0.000000	0.000000
3	0.941819	-0.003702	0.939975	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP		LossQ
1

Bifurcated agents:   0%|          | 0/1024 [00:02<?, ?it/s]


Minimizer: [0.    0.125 0.25  0.125 0.875 0.875 0.875 0.5   0.5   0.5   0.625 0.375
 0.875 0.5   0.5   0.5   0.5   0.5   0.5   0.625 0.375 0.5   0.5   0.5
 0.5   0.    0.    0.    0.    0.    0.   ]
||h(x)|| = 0.5673646325128939
[rho-check] ||h_old||=3.541e+00, ||h_new||=5.674e-01, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-20 12:12:54
Iteration: 1
rho: 50
alpha: 9.6976536
objective_value: 0
feasible: False
max_abs_h: 4.113832469783e-01
l2_norm_h: 5.673646325129e-01
lambda_inf_norm: 3.016388358506e+01
lambda_l2_norm: 4.086992022083e+01
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.000038	-0.000102	1.001769	0.032271	0.739082	0.000000	0.000000
2	0.962861	0.010585	0.961246	0.000000	-0.735016	0.000000	0.000000
3	0.968443	-0.013238	0.967380	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP	

...

Minimizer: [0.25  0.125 0.375 0.    0.875 0.75  0.75  0.5   0.5   0.5   0.375 0.125
 0.    0.625 0.5   0.5   0.5   0.5   0.5   0.875 0.125 0.625 0.5   0.5
 0.5   0.75  0.75  0.    0.    0.    0.   ]
||h(x)|| = 0.6828112245302508
[rho-check] ||h_old||=2.110e+00, ||h_new||=6.828e-01, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-20 17:52:06
Iteration: 597
rho: 50
alpha: 1.0964782e-07
objective_value: 0
feasible: False
max_abs_h: 5.390141704184e-01
l2_norm_h: 6.828112245303e-01
lambda_inf_norm: 9.428166539748e+01
lambda_l2_norm: 1.331554761795e+02
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.999955	-0.000059	1.000529	0.000000	0.612915	0.000000	0.000000
2	0.971846	0.016926	0.973419	0.313232	-0.572716	0.000000	0.000000
3	0.969226	-0.018472	0.967379	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		

Bifurcated agents:   0%|          | 0/1024 [00:02<?, ?it/s]


Minimizer: [0.125 0.    0.125 0.125 0.875 0.875 0.875 0.5   0.5   0.5   0.625 0.375
 0.625 0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5
 0.5   0.125 0.25  0.    0.    0.    0.   ]
||h(x)|| = 2.091427431809353
[rho-check] ||h_old||=6.828e-01, ||h_new||=2.091e+00, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-20 17:52:40
Iteration: 598
rho: 50
alpha: 1.0633266e-07
objective_value: 0
feasible: False
max_abs_h: 1.504105597470e+00
l2_norm_h: 2.091427431809e+00
lambda_inf_norm: 9.428166555741e+01
lambda_l2_norm: 1.331554764015e+02
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.999689	-0.000256	1.000560	0.000000	-1.519990	0.000000	0.000000
2	1.078907	-0.016341	1.077022	0.377794	1.559122	0.000000	0.000000
3	1.024289	-0.034317	1.023810	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		

Bifurcated agents:   0%|          | 0/1024 [00:02<?, ?it/s]


Minimizer: [0.    0.125 0.25  0.125 0.875 0.875 0.875 0.5   0.5   0.5   0.    0.625
 0.125 0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.5   0.625 0.5   0.5
 0.5   0.    0.    0.    0.    0.    0.   ]
||h(x)|| = 0.0651398388687014
[rho-check] ||h_old||=2.091e+00, ||h_new||=6.514e-02, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-20 17:53:17
Iteration: 599
rho: 50
alpha: 1.0311773e-07
objective_value: 0
feasible: False
max_abs_h: 5.165617200000e-02
l2_norm_h: 6.513983886870e-02
lambda_inf_norm: 9.428166555209e+01
lambda_l2_norm: 1.331554763987e+02
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.000227	-0.000194	0.985617	0.000000	0.057098	0.000000	0.000000
2	0.999930	0.006831	0.999086	0.259220	-0.055757	0.000000	0.000000
3	0.986016	-0.020518	0.987094	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		

In [6]:
Lag

0.00375*P_G0**2 + 0.278200168156819*P_G0 + 0.0175*P_G1**2 + 0.0157547129319766*P_G1 - 29.8493633224316*P_ij0 + 29.9746108177359*P_ij1 + 0.816389865909716*P_ij2 - 0.706185802988601*P_ij3 + 0.756489419576001*P_ij4 - 0.849900895055773*P_ij5 + 0.0136572917523153*Q_G0 + 0.0164691044884147*Q_G1 - 252.467648346265*Q_ij0 + 238.627093875613*Q_ij1 - 0.899595417866211*Q_ij2 + 0.275746097739299*Q_ij3 + 0.326731325460938*Q_ij4 - 0.815068405438657*Q_ij5 - 94.2816655574149*S_tot_sq0 - 93.5455694326835*S_tot_sq1 - 2.48726276635139*S_tot_sq2 - 2.39260063894758*S_tot_sq3 - 2.38194570224713*S_tot_sq4 - 2.4324341246754*S_tot_sq5 + 10000.0*V_I0**2 + 2.53213603260473*V_I0 - 4.10397611598224*V_I1 + 0.7170860544584*V_I2 - 105.880563117351*V_R0 + 119.508045748219*V_R1 - 5.44596239953777*V_R2 - 0.0158173141887125*V_sq0 - 0.341174712882471*V_sq1 - 0.105623494580877*V_sq2 + 25.0*(20.0*V_R0 - 20.0)**2 + 25.0*(1.0*P_G0 - 1.0*P_ij0 - 1.0*P_ij2)**2 + 25.0*(1.0*P_G1 - 1.0*P_ij1 - 1.0*P_ij4)**2 + 25.0*(-1.0*P_ij3 - 1.0

In [7]:
model.P_D

array([0. , 0. , 0.3])